#IT Service Ticket Classification — Data Preparation Pipeline

## Introduction

This notebook focuses on preparing a real-world IT service ticket dataset for a multi-class classification problem.

The objective is to transform raw ticket data into a clean, structured, and model-ready dataset by applying standard data preprocessing techniques. This includes handling missing values, removing duplicates, filtering irrelevant categories, and selecting meaningful classes aligned with real IT support operations.

The resulting dataset will be used in subsequent stages to build and deploy a machine learning model capable of automatically classifying incoming support tickets.

This step is critical to ensure data quality, model performance, and reproducibility within a production-oriented machine learning workflow.

### Dataset Preparation (Production Style)

In [1]:
# Import pandas for DataFrame manipulation
import pandas as pd

# Define path to the raw dataset file
raw_data_path = "../data/raw/raw_tickets.csv"

# Load dataset into a pandas DataFrame
df = pd.read_csv(raw_data_path)

# Display first 5 rows to inspect structure
df.head()

,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous


In [2]:
# Display dataset dimensions (rows, columns)
df.shape

(47837, 2)

In [3]:
# Display dataset structure: column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47837 entries, 0 to 47836
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Document     47837 non-null  object
 1   Topic_group  47837 non-null  object
dtypes: object(2)
memory usage: 747.6+ KB


In [4]:
# List all column names in the dataset
df.columns

Index(['Document', 'Topic_group'], dtype='object')

In [5]:
# Check for missing values in each column
df.isnull().sum()

Document       0
Topic_group    0
dtype: int64

In [6]:
# Count duplicated rows in the dataset
df.duplicated().sum()

np.int64(0)

In [7]:
# Inspect original category distribution
df["Topic_group"].value_counts()

Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64

### Select and Rename Columns

In [8]:
# Select only relevant columns for the classification task
df = df[["Document", "Topic_group"]].copy()

# Rename columns to standardized, production-friendly names
df.columns = ["ticket_text", "category"]

# Preview updated dataset
df.head()

,ticket_text,category
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous


### Data Cleaning

In [9]:
# Remove rows with missing values in key columns
df = df.dropna(subset=["ticket_text", "category"])

# Remove duplicated rows
df = df.drop_duplicates()

# Ensure text column is string type and strip leading/trailing spaces
df["ticket_text"] = df["ticket_text"].astype(str).str.strip()

# Ensure category column is string type and strip leading/trailing spaces
df["category"] = df["category"].astype(str).str.strip()

# Remove very short tickets (low information content)
df = df[df["ticket_text"].str.len() > 20]

# Check dataset size after cleaning
df.shape

(47780, 2)

In [10]:
# Inspect category distribution after cleaning
df["category"].value_counts()

category
Hardware                 13596
HR Support               10903
Access                    7111
Miscellaneous             7054
Storage                   2777
Purchase                  2460
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64

### Define Final Classes

In [11]:
# Define final categories based on relevant and interpretable IT support classes
final_classes = [
    "Hardware",
    "Access",
    "Storage",
    "Purchase",
    "Administrative rights",
    "HR Support"
]

# Filter dataset to keep only selected categories
df_clean = df[df["category"].isin(final_classes)].copy()

# Check distribution of the filtered dataset
df_clean["category"].value_counts()

category
Hardware                 13596
HR Support               10903
Access                    7111
Storage                   2777
Purchase                  2460
Administrative rights     1760
Name: count, dtype: int64

In [12]:
# Check final dataset size
df_clean.shape

(38607, 2)

### Text Length Analysis (Basic Quality Check)

In [13]:
# Create auxiliary column to analyze text length
df_clean["text_length"] = df_clean["ticket_text"].str.len()

# Generate descriptive statistics of text length by category
df_clean.groupby("category")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
Access,7111.0,242.779778,332.515913,21.0,96.00,147.0,254.00,6363.0
Administrative rights,1760.0,339.180682,440.187731,21.0,120.75,201.0,363.75,6880.0
HR Support,10903.0,249.507842,284.317391,21.0,101.00,159.0,267.00,5595.0
Hardware,13596.0,375.959988,516.465209,21.0,127.00,206.0,393.00,7015.0
Purchase,2460.0,244.899593,183.060678,37.0,155.00,209.0,263.00,2827.0
Storage,2777.0,233.346417,292.040889,27.0,96.00,142.0,246.00,3021.0


In [14]:
# Remove auxiliary column before saving final dataset
df_clean = df_clean.drop(columns=["text_length"])

### Final Validation

In [15]:
# Confirm no missing values remain
df_clean.isnull().sum()

ticket_text    0
category       0
dtype: int64

In [16]:
# Confirm no duplicated rows remain
df_clean.duplicated().sum()

np.int64(0)

In [17]:
# Confirm final class distribution
df_clean["category"].value_counts()

category
Hardware                 13596
HR Support               10903
Access                    7111
Storage                   2777
Purchase                  2460
Administrative rights     1760
Name: count, dtype: int64

### Export Clean Dataset 

In [18]:
# Define output path for cleaned dataset
clean_data_path = "../data/processed/clean_tickets.csv"

# Export cleaned dataset to CSV format (without index)
df_clean.to_csv(clean_data_path, index=False)